In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
import os
from tqdm import tqdm

IDS_CHUNKS_DIR = "ids/"
VISUAL_CHUNKS_DIR = "embeddings/"        # folder with all 768 chunks
CONTEXT_IDS_PATH  = "BRAIN/context_photo_ids.npy"
CSV_PATH          = "usa_europe_geotagged.csv"
OUT_DIR           = "filtered_embeddings_v2"
os.makedirs(OUT_DIR, exist_ok=True)

TOTAL_CHUNKS = 768

In [2]:
context_ids    = np.load(CONTEXT_IDS_PATH, allow_pickle=True)
context_id_set = set(context_ids)

total_matched = 0
emb_out = open(os.path.join(OUT_DIR, "visual_embeddings_filtered.npy"), 'wb')
ids_out = []

# We'll collect all matched IDs (lightweight, just integers)
for i in tqdm(range(TOTAL_CHUNKS), desc="Processing chunks"):
    ids  = np.load(os.path.join(IDS_CHUNKS_DIR, f"ids_chunk_{i}.npy"),        allow_pickle=True)
    embs = np.load(os.path.join(VISUAL_CHUNKS_DIR, f"embeddings_chunk_{i}.npy"), mmap_mode='r')

    mask = np.array([pid in context_id_set for pid in ids])

    if mask.any():
        matched_embs = embs[mask]
        matched_ids  = ids[mask]

        # Save each chunk's embeddings separately
        np.save(os.path.join(OUT_DIR, f"filtered_emb_chunk_{i}.npy"), matched_embs)
        ids_out.append(matched_ids)
        total_matched += len(matched_ids)

emb_out.close()

# Save all matched IDs as one file (just IDs, very small)
filtered_visual_ids = np.concatenate(ids_out)
np.save(os.path.join(OUT_DIR, "visual_photo_ids_filtered.npy"), filtered_visual_ids)

print(f"Total matched : {total_matched:,}")
print(f"Chunks saved  : {TOTAL_CHUNKS} (only non-empty chunks written)")

Processing chunks: 100%|█████████████████████████████████████████████████████████████| 768/768 [29:51<00:00,  2.33s/it]


Total matched : 4,439,390
Chunks saved  : 768 (only non-empty chunks written)


In [7]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
import os

FILTERED_DIR = "brain/"
CSV_PATH     = "usa_europe_geotagged.csv"

# Load the full filtered IDs
filtered_visual_ids = np.load(os.path.join(FILTERED_DIR, "visual_photo_ids_filtered.npy"), allow_pickle=True)
print(f"Total filtered IDs: {len(filtered_visual_ids):,}")

# Load matching rows from CSV
print("Loading coordinates from CSV...")
df = pd.read_csv(CSV_PATH, usecols=['photoid', 'latitude', 'longitude'], low_memory=False)
df = df[df['photoid'].isin(set(filtered_visual_ids))].reset_index(drop=True)
print(f"Coordinates loaded: {len(df):,} rows")

Total filtered IDs: 4,439,390
Loading coordinates from CSV...
Coordinates loaded: 4,439,390 rows


In [11]:
print("Running DBSCAN clustering...")

coords_rad = np.radians(df[['latitude', 'longitude']].values)

db = DBSCAN(eps=100/6371000, min_samples=3, algorithm='ball_tree', metric='haversine', n_jobs=-1)
df['poi_id'] = db.fit_predict(coords_rad)

print(f"POIs found  : {df[df['poi_id'] >= 0]['poi_id'].nunique():,}")
print(f"Noise points: {(df['poi_id'] == -1).sum():,}")

Running DBSCAN clustering...
POIs found  : 166,850
Noise points: 504,923


In [9]:
df_clean = df[df['poi_id'] >= 0].reset_index(drop=True)
print(f"Retained: {len(df_clean):,} photos across {df_clean['poi_id'].nunique():,} POIs")

Retained: 3,744,663 photos across 201,623 POIs


In [12]:
df_clean.to_csv(os.path.join(FILTERED_DIR, "photo_poi_mapping.csv"), index=False)
print("Saved: photo_poi_mapping.csv")
print(df_clean.head())

Saved: photo_poi_mapping.csv
   photoid   longitude   latitude  poi_id
0    29872  -93.440796  38.760236       0
1    29873  -93.440796  38.760236       0
2    29874  -93.440796  38.760236       0
3    29875  -93.440796  38.760236       0
4    30431 -122.351903  47.619673       1


In [16]:
print(len(df_clean))

3744663
